# Scanpy scRNA-seq pipeline on Smillie 2019 UC data

**Dataset:** Smillie et al. 2019, *Cell* â€” ulcerative colitis mucosal scRNA-seq ([SCP259](https://singlecell.broadinstitute.org/single_cell/study/SCP259)).

**Goal:** end-to-end Scanpy pipeline â€” QC, clustering, manual cell-type annotation with a focus on the myeloid compartment in healthy vs inflamed colonic mucosa.

**Structure**

1. Imports and settings
2. Load data (with a PBMC3k fallback so the notebook runs end-to-end before Smillie is downloaded)
3. QC metrics
4. Cell + gene filtering
5. Doublet detection (Scrublet)
6. Normalisation + log1p
7. Highly variable genes
8. Scaling + PCA
9. Neighbours + UMAP
10. Leiden clustering
11. Marker genes
12. Broad cell-type annotation
13. Myeloid zoom-in (subcluster + annotate)
14. Final figure: UMAP split by condition
15. Save processed AnnData

**Key object:** `adata` â€” an [`AnnData`](https://anndata.readthedocs.io/) â€” holds the count matrix and all annotations.

| AnnData slot | Contains | Seurat equivalent |
|---|---|---|
| `adata.X` | Expression matrix, cells Ã— genes | `seurat_obj@assays$RNA@counts` / `@data` |
| `adata.obs` | Per-cell metadata (DataFrame) | `seurat_obj@meta.data` |
| `adata.var` | Per-gene metadata (DataFrame) | `seurat_obj@assays$RNA@meta.features` |
| `adata.obsm` | Multi-dim embeddings (PCA, UMAP) | `seurat_obj@reductions` |
| `adata.uns` | Unstructured (clustering params, colours) | Misc. `@misc` |
| `adata.layers` | Alternative matrices (e.g. raw counts alongside normalised) | multiple assays / slots |

## 1. Imports and settings

Conventional Scanpy import aliases: `sc` for `scanpy`, `np` for `numpy`, `pd` for `pandas`. `sc.settings` is a global config object (verbosity, default figure size).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import anndata as ad

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=80, facecolor="white")
sc.logging.print_header()

## 2. Load data

The Smillie data lives in `data/smillie/` (gitignored). Set `USE_PBMC_FALLBACK = True` to run the whole notebook on the built-in PBMC3k dataset instead â€” useful for sanity-checking the pipeline before the full dataset is ready.

**R/Seurat equivalent:** `Read10X("data/filtered_gene_bc_matrices/")` + `CreateSeuratObject(...)`.

In [ ]:
USE_PBMC_FALLBACK = True  # flip to False once Smillie is downloaded

if USE_PBMC_FALLBACK:
    # 2,700 PBMCs from 10X Genomics â€” ships with Scanpy, good for pipeline dry-run.
    adata = sc.datasets.pbmc3k()
    adata.obs["condition"] = "healthy"  # placeholder so split-by-condition plots work
else:
    # TODO: fill in once Smillie download is in place.
    # Broad SCP259 gives .mtx + barcodes.tsv + features.tsv + cell metadata TSV.
    # Expected path layout:
    #   data/smillie/matrix.mtx.gz
    #   data/smillie/barcodes.tsv.gz
    #   data/smillie/features.tsv.gz
    #   data/smillie/all.meta2.txt  (cell-level metadata incl. Health vs Inflamed)
    raise NotImplementedError("Smillie loader â€” complete once data is downloaded.")

adata.var_names_make_unique()  # Smillie (and 10X in general) can have duplicate gene symbols; safe to always call.
adata

## 3. QC metrics

Flag mitochondrial and ribosomal gene sets, then compute per-cell QC metrics. Three metrics drive filtering:

- **`n_genes_by_counts`** â€” genes detected per cell. Very low â†’ empty droplet / dying cell. Very high â†’ possible doublet.
- **`total_counts`** â€” UMIs per cell. Same logic.
- **`pct_counts_mt`** â€” % mitochondrial reads. High â†’ stressed / dying cell.

**R/Seurat equivalent:** `seurat_obj[["percent.mt"]] <- PercentageFeatureSet(seurat_obj, pattern = "^MT-")`.

**Parameter note:** human gene symbols use `MT-` prefix; mouse uses `mt-` (lowercase). Smillie is human â†’ `MT-`.

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))

sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt", "ribo"],
    percent_top=None,
    log1p=False,
    inplace=True,
)

adata.obs[["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo"]].describe()

In [ ]:
# Violin plots of the three main QC metrics â€” look for bimodality and long tails.
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
# Scatter plots of pairwise QC metrics â€” helps pick filter thresholds visually.
sc.pl.scatter(adata, x="total_counts", y="pct_counts_mt")
sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts")

## 4. Cell and gene filtering

Drop obvious failures. Thresholds are a **judgment call** â€” defend them based on the QC plots above, not as universal rules.

- `min_genes=200`: standard first-pass floor for 10X data.
- `min_cells=3`: gene must be seen in â‰¥3 cells to be kept â€” removes noise genes.
- `pct_counts_mt < 20`: permissive ceiling for colonic mucosa (epithelial cells tolerate more MT%); 10% is typical for PBMCs.

**R/Seurat equivalent:** `subset(seurat_obj, subset = nFeature_RNA > 200 & percent.mt < 20)`.

In [ ]:
print(f"Before filtering: {adata.n_obs} cells, {adata.n_vars} genes")

sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

adata = adata[adata.obs["pct_counts_mt"] < 20, :].copy()

print(f"After filtering:  {adata.n_obs} cells, {adata.n_vars} genes")

## 5. Doublet detection (Scrublet)

Scrublet simulates synthetic doublets by averaging random cell pairs, then scores each real cell against that distribution. Cells with high `doublet_score` are likely captured doublets and should be removed.

**Note:** run Scrublet **per sample / per 10X lane**, not on pooled data â€” doublets are a within-droplet artefact, not a biological one. With PBMC3k there's a single sample, so one call is fine. With Smillie's multi-sample design we'd loop over samples.

In [ ]:
# Using scanpy's built-in reimplementation (sc.pp.scrublet) rather than the
# external `scrublet` package — avoids a Windows C++ build dependency and
# keeps the environment slim. Same algorithm.
sc.pp.scrublet(adata)

sc.pl.violin(adata, "doublet_score", jitter=0.4)
print(adata.obs["predicted_doublet"].value_counts())

adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs} cells")

## 6. Normalisation + log1p

Two-step normalisation, same as Seurat's `NormalizeData()` with default `LogNormalize`:

1. `normalize_total(target_sum=1e4)` â€” scale each cell so its counts sum to 10,000 ("CP10k"). Corrects for sequencing depth differences.
2. `log1p` â€” natural log of `(x + 1)`. Stabilises variance so highly-expressed genes don't dominate.

**Keep a copy of raw counts in `adata.layers["counts"]`** â€” needed later for things like DE with `diffxpy` or re-normalisation. Cheap insurance.

In [ ]:
adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

## 7. Highly variable genes (HVGs)

Restrict downstream analysis (PCA, clustering) to the genes that vary most across cells. Removes housekeeping / constant-expression genes that add noise but no biological signal.

`flavor="seurat"` picks genes by binned mean/dispersion â€” the same method `FindVariableFeatures(selection.method = "vst")` approximates. `n_top_genes=2000` is the conventional default.

`adata.raw = adata` freezes the full log-normalised matrix before we subset to HVGs â€” so we can still plot/score *any* gene (e.g. marker genes like `CD14`) even though downstream clustering uses HVGs only.

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat")
sc.pl.highly_variable_genes(adata)

adata.raw = adata
adata = adata[:, adata.var["highly_variable"]].copy()

## 8. Scaling + PCA

- `sc.pp.scale(..., max_value=10)` â€” per-gene z-score with clipping. Clipping at 10 SDs prevents rare outlier cells from dominating PCs.
- `sc.tl.pca` â€” linear dimensionality reduction. We keep ~50 PCs and use them as the input to the neighbour graph.

**Elbow plot** (variance explained per PC) helps pick how many PCs to feed into `neighbors`. For immune/epithelial datasets 30â€“50 is typical.

In [ ]:
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver="arpack")
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

## 9. Neighbours + UMAP

- `sc.pp.neighbors` builds the KNN graph on the PCA embedding â€” the backbone of both UMAP and Leiden.
- `sc.tl.umap` projects that graph into 2D for visualisation only. **UMAP is not used for clustering** â€” Leiden runs on the neighbour graph directly.

**R/Seurat equivalent:** `FindNeighbors()` + `RunUMAP()`.

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=40)
sc.tl.umap(adata)
sc.pl.umap(adata, color=["total_counts", "pct_counts_mt"])

## 10. Leiden clustering

Leiden is a community-detection algorithm on the neighbour graph. `resolution` controls cluster granularity â€” higher = more, smaller clusters.

**Judgment call:** there is no "right" resolution. Run 2â€“3 and pick the one whose clusters match your biological expectations (and where marker genes are clean). Resolutions 0.5â€“1.0 are typical starting points.

In [ ]:
for res in [0.3, 0.5, 0.8]:
    sc.tl.leiden(adata, resolution=res, key_added=f"leiden_{res}")

sc.pl.umap(adata, color=["leiden_0.3", "leiden_0.5", "leiden_0.8"], legend_loc="on data")

In [ ]:
# Pick one resolution to carry forward. Revisit this after looking at marker genes.
adata.obs["leiden"] = adata.obs["leiden_0.5"]

## 11. Marker genes per cluster

One-vs-rest differential expression per cluster. Wilcoxon rank-sum is the standard non-parametric choice for log-normalised data â€” same default as Seurat's `FindAllMarkers()`.

Top markers guide manual annotation (next section).

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon")
sc.pl.rank_genes_groups(adata, n_genes=15, sharey=False)

In [ ]:
# Top markers as a tidy DataFrame â€” easier to search than the plot.
top_markers = pd.DataFrame({
    f"cluster_{c}": adata.uns["rank_genes_groups"]["names"][c][:10]
    for c in adata.obs["leiden"].cat.categories
})
top_markers

## 12. Broad cell-type annotation

Match clusters to canonical lineage markers. Strategy: plot a curated marker set as a dot plot, then write a `cluster â†’ cell_type` dictionary.

**Canonical markers for colonic mucosa** (Smillie) â€” also valid for PBMC3k for the immune subset:

| Lineage | Markers |
|---|---|
| Epithelial | `EPCAM`, `KRT8`, `KRT18` |
| Stromal / fibroblast | `COL1A1`, `COL3A1`, `DCN` |
| Endothelial | `PECAM1`, `VWF` |
| T cell | `CD3D`, `CD3E`, `CD8A`, `CD4` |
| B cell | `MS4A1`, `CD79A` |
| Plasma | `MZB1`, `JCHAIN` |
| Myeloid (broad) | `LYZ`, `CD14`, `FCGR3A`, `CD68`, `S100A8`, `S100A9` |
| Mast | `TPSAB1`, `KIT` |

**Caveat for PBMC3k fallback:** no epithelial / stromal / endothelial lineages â€” only immune clusters will light up.

In [ ]:
marker_genes = {
    "Epithelial": ["EPCAM", "KRT8", "KRT18"],
    "Stromal": ["COL1A1", "COL3A1", "DCN"],
    "Endothelial": ["PECAM1", "VWF"],
    "T cell": ["CD3D", "CD3E", "CD8A", "CD4"],
    "B cell": ["MS4A1", "CD79A"],
    "Plasma": ["MZB1", "JCHAIN"],
    "Myeloid": ["LYZ", "CD14", "FCGR3A", "CD68", "S100A8", "S100A9"],
    "Mast": ["TPSAB1", "KIT"],
}

# Keep only markers actually present (PBMC3k won't have all of them).
marker_genes = {
    k: [g for g in v if g in adata.raw.var_names]
    for k, v in marker_genes.items()
}
marker_genes = {k: v for k, v in marker_genes.items() if v}

sc.pl.dotplot(adata, marker_genes, groupby="leiden", standard_scale="var")

In [ ]:
# Fill in once the dot plot has been inspected. Clusters not in the dict default to "Unknown".
cluster_to_celltype = {
    # "0": "T cell",
    # "1": "Myeloid",
    # "2": "B cell",
    # ...
}

adata.obs["cell_type"] = (
    adata.obs["leiden"].map(cluster_to_celltype).fillna("Unknown").astype("category")
)
sc.pl.umap(adata, color="cell_type", legend_loc="on data")

## 13. Myeloid zoom-in

Subset to cells annotated as `Myeloid`, then re-run HVG â†’ PCA â†’ neighbours â†’ UMAP â†’ Leiden on that compartment alone. This teases apart subpopulations (monocytes, macrophages, cDC1, cDC2, pDC) that look like one blob at the whole-dataset level.

Common myeloid subtype markers:

| Subtype | Markers |
|---|---|
| Classical monocyte | `CD14`, `S100A8`, `S100A9`, `VCAN` |
| Non-classical monocyte | `FCGR3A` (CD16), `MS4A7`, `CDKN1C` |
| Macrophage (gut-resident) | `C1QA`, `C1QB`, `C1QC`, `APOE` |
| cDC1 | `CLEC9A`, `XCR1`, `IRF8`, `BATF3` |
| cDC2 | `CLEC10A`, `CD1C`, `FCER1A` |
| pDC | `LILRA4`, `GZMB`, `IRF7` |

**Skip this section in PBMC3k fallback** â€” only 2700 cells, myeloid subset will be tiny and noisy.

In [ ]:
if "Myeloid" in adata.obs["cell_type"].cat.categories:
    # Start from the full log-normalised matrix in .raw, subset to myeloid cells.
    mye = adata.raw.to_adata()[adata.obs["cell_type"] == "Myeloid"].copy()
    sc.pp.highly_variable_genes(mye, n_top_genes=2000, flavor="seurat")
    mye = mye[:, mye.var["highly_variable"]].copy()
    sc.pp.scale(mye, max_value=10)
    sc.tl.pca(mye, svd_solver="arpack")
    sc.pp.neighbors(mye, n_neighbors=10, n_pcs=20)
    sc.tl.umap(mye)
    sc.tl.leiden(mye, resolution=0.5, key_added="leiden_mye")
    sc.pl.umap(mye, color="leiden_mye", legend_loc="on data")

    mye_markers = {
        "Classical mono": ["CD14", "S100A8", "S100A9", "VCAN"],
        "Non-classical mono": ["FCGR3A", "MS4A7", "CDKN1C"],
        "Macrophage": ["C1QA", "C1QB", "C1QC", "APOE"],
        "cDC1": ["CLEC9A", "XCR1", "IRF8", "BATF3"],
        "cDC2": ["CLEC10A", "CD1C", "FCER1A"],
        "pDC": ["LILRA4", "GZMB", "IRF7"],
    }
    # Restrict to markers actually present in the myeloid subset's gene list.
    mye_markers = {k: [g for g in v if g in mye.var_names] for k, v in mye_markers.items()}
    mye_markers = {k: v for k, v in mye_markers.items() if v}
    sc.pl.dotplot(mye, mye_markers, groupby="leiden_mye", standard_scale="var")
else:
    print("No Myeloid cluster — skip this section (expected in PBMC3k fallback run).")

## 14. Final figure: UMAP split by condition

One clean summary figure â€” UMAP coloured by cell type, faceted by healthy vs inflamed. This is the slide-worthy figure.

In [ ]:
sc.pl.umap(
    adata,
    color="cell_type",
    groups=None,
    save="_celltype.png",
)

if adata.obs["condition"].nunique() > 1:
    sc.pl.umap(
        adata,
        color="cell_type",
        ncols=2,
        save="_celltype_by_condition.png",
    )
    # facet manually by condition â€” Scanpy's native `groupby` faceting is limited.
    import matplotlib.pyplot as plt
    conditions = adata.obs["condition"].cat.categories if hasattr(adata.obs["condition"], "cat") else adata.obs["condition"].unique()
    fig, axes = plt.subplots(1, len(conditions), figsize=(6 * len(conditions), 5))
    if len(conditions) == 1:
        axes = [axes]
    for ax, cond in zip(axes, conditions):
        sc.pl.umap(adata[adata.obs["condition"] == cond], color="cell_type", ax=ax, show=False, title=cond)
    plt.tight_layout()

## 15. Save processed AnnData

Serialise the fully-annotated object to `data/processed/` for reuse. HDF5 format â€” portable, fast, readable from both Python (`sc.read_h5ad`) and R (via `zellkonverter`).

In [ ]:
import os
os.makedirs("../data/processed", exist_ok=True)
out = "../data/processed/smillie_annotated.h5ad" if not USE_PBMC_FALLBACK else "../data/processed/pbmc3k_annotated.h5ad"
adata.write_h5ad(out)
print(f"Wrote {out}")